# Module 6 Worksheet — MCP: Building a Minimal Server
Requires: `pip install mcp --break-system-packages`. This worksheet writes a server file to disk and a client script — run the client in a separate cell/process since MCP servers run as subprocesses.

**Corrected in this version:** section 3's agent now uses `ask()` from the setup cell instead of re-importing and calling `multimodal_chat()` directly.

## 1. A minimal MCP server exposing one tool
This cell WRITES a server file — it doesn't run it directly.

In [ ]:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("capstone-tools")

@mcp.tool()
def search_documents(query: str) -> str:
    """Search a tiny in-memory knowledge base and return the best match."""
    facts = [
        "MCP standardizes how LLMs call external tools through a client-server interface.",
        "RAG combines a retriever and a generator to ground LLM answers in context.",
    ]
    best = max(facts, key=lambda f: len(set(query.lower().split()) & set(f.lower().split())))
    return best

@mcp.tool()
def calculator(expression: str) -> str:
    """Evaluate a basic math expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("mcp_server.py", "w") as f:
    f.write(server_code)
print("Wrote mcp_server.py")

## 2. A minimal MCP client connecting over stdio
This spawns `mcp_server.py` as a subprocess and calls its tools — this IS the Host/Client/Server diagram from `diagrams.md` running live.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_client():
    server_params = StdioServerParameters(command="python", args=["mcp_server.py"])
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Tools exposed by server:", [t.name for t in tools.tools])

            result = await session.call_tool("search_documents", {"query": "What is MCP?"})
            print("search_documents result:", result.content)

            result2 = await session.call_tool("calculator", {"expression": "12 * 7"})
            print("calculator result:", result2.content)

await run_client()  # in Jupyter, top-level await works; in a plain script use asyncio.run(run_client())

## 3. Connecting your in-house model as the agent deciding WHICH tool to call
This is the bridge from Module 5's manual ReAct loop to MCP-backed tools — same decision logic, but the tool execution now happens in a separate process. Uses `ask()` from the setup cell (run that cell first if you haven't this session).

In [ ]:
import json

async def mcp_agent(question, max_steps=3):
    server_params = StdioServerParameters(command="python", args=["mcp_server.py"])
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            tool_desc = "\n".join(f"- {t.name}: {t.description}" for t in tools.tools)

            system = f"""You have these MCP tools available:
{tool_desc}

Respond with ONLY one JSON object:
{{"action": "<tool_name>", "action_input": {{...}}}}
or
{{"action": "final_answer", "action_input": "<answer>"}}
"""
            transcript = f"Question: {question}"
            for step in range(max_steps):
                raw = ask(system, transcript, max_tokens=150)
                action = json.loads(raw)
                if action["action"] == "final_answer":
                    return action["action_input"]
                result = await session.call_tool(action["action"], action["action_input"])
                transcript += f"\nAction: {action}\nObservation: {result.content}"
            return "Max steps reached."

answer = await mcp_agent("What is MCP, and what is 12 * 7?")
print("FINAL:", answer)

## Teaser exercise
Run the server with `transport="sse"` instead of `"stdio"` and connect with an HTTP-based client. What changes in your client code, and what stays the same? (Hint: the tool-calling logic above shouldn't need to change at all — that's the point of the protocol abstraction.)